# Silver Lecture 2 — Markov Decision Processes

*Companion notes, in slide order*

**Big picture:** almost every RL problem can be written as an MDP. This lecture builds up from Markov processes → Markov reward processes → Markov decision processes, then defines value functions, the Bellman equations, and optimality.

## 1. Markov processes

**Markov property.** A state is Markov if the future is independent of the past given the present:

$$\mathbb{P}[S_{t+1} \mid S_t] = \mathbb{P}[S_{t+1} \mid S_1, \dots, S_t]$$

The state captures all relevant information from the history; once you know it, the history can be thrown away.

**State transition probability** from state $s$ to successor $s'$:

$$\mathcal{P}_{ss'} = \mathbb{P}[S_{t+1} = s' \mid S_t = s]$$

**State transition matrix** (row = from, column = to; each row sums to 1):

$$\mathcal{P} = \text{from}\ \overset{\text{to}}{\begin{bmatrix} \mathcal{P}_{11} & \cdots & \mathcal{P}_{1n} \\ \vdots & & \vdots \\ \mathcal{P}_{n1} & \cdots & \mathcal{P}_{nn} \end{bmatrix}}$$

**Markov process (Markov chain):** a tuple $\langle \mathcal{S}, \mathcal{P} \rangle$, a memoryless random sequence of states $S_1, S_2, \dots$

*Student example:* states Class 1–3, Pass, Pub, Facebook, Sleep. Sampling episodes gives sequences like C1 C2 C3 Pass Sleep. Sleep is a terminal state.

## 2. Markov reward processes (MRP)

An MRP adds rewards and a discount: $\langle \mathcal{S}, \mathcal{P}, \mathcal{R}, \gamma \rangle$.

**Reward function** (expected immediate reward):

$$\mathcal{R}_s = \mathbb{E}[R_{t+1} \mid S_t = s]$$

**Discount** $\gamma \in [0, 1]$.

**Return** $G_t$: total discounted reward from time $t$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

- $\gamma$ near 0: "myopic", cares about the next reward.
- $\gamma$ near 1: "far-sighted".
- A reward $k+1$ steps ahead is worth $\gamma^k R$ now.

**Why discount?**

- Mathematically convenient; keeps returns finite in cyclic processes.
- Uncertainty about the future may not be fully represented.
- Financial analogy: immediate reward earns more interest.
- Humans/animals prefer immediate reward.
- $\gamma = 1$ is fine if all sequences terminate (episodic tasks).

**Value function** of an MRP: the expected return from state $s$:

$$v(s) = \mathbb{E}[G_t \mid S_t = s]$$

Note $G_t$ is random (one sample); $v(s)$ is its expectation.

### Bellman equation for MRPs

Split the return into the immediate reward plus the discounted value of the next state:

$$v(s) = \mathbb{E}[R_{t+1} + \gamma\, v(S_{t+1}) \mid S_t = s]$$

$$v(s) = \mathcal{R}_s + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}\, v(s')$$

Reward of the current state + discounted, probability-weighted rewards of the next states

*Backup diagram:* from $s$, average over the possible next states $s'$.

**Matrix form:** $v = \mathcal{R} + \gamma \mathcal{P} v$, which is linear, so it can be solved directly:

$$v = (I - \gamma \mathcal{P})^{-1} \mathcal{R}$$

- Cost is $O(n^3)$ for $n$ states, so only practical for small MRPs.
- Large MRPs use iterative methods: dynamic programming (L3), Monte-Carlo evaluation (L4), temporal-difference learning (L4).

## 3. Markov decision processes (MDP)

An MDP adds actions: $\langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$. All states are Markov.

$$\mathcal{P}_{ss'}^a = \mathbb{P}[S_{t+1} = s' \mid S_t = s, A_t = a] \qquad \mathcal{R}_s^a = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$$

**Policy** $\pi$: a distribution over actions given states:

$$\pi(a \mid s) = \mathbb{P}[A_t = a \mid S_t = s]$$

- Depends only on the current state, not the history (because the state is Markov).
- Stationary: $A_t \sim \pi(\cdot \mid S_t)$ for all $t$.

**An MDP plus a fixed policy is an MRP:**

$$\mathcal{P}_{ss'}^{\pi} = \sum_{a \in \mathcal{A}} \pi(a \mid s)\, \mathcal{P}_{ss'}^a \qquad \mathcal{R}_s^{\pi} = \sum_{a \in \mathcal{A}} \pi(a \mid s)\, \mathcal{R}_s^a$$

### Value functions

**State-value function:** expected return starting in $s$ and following $\pi$:

$$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$$

**Action-value function:** expected return starting in $s$, taking action $a$, then following $\pi$:

$$q_\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$$

### Bellman expectation equations

Same idea as before, now with actions:

$$v_\pi(s) = \mathbb{E}_\pi[R_{t+1} + \gamma\, v_\pi(S_{t+1}) \mid S_t = s]$$

$$q_\pi(s,a) = \mathbb{E}_\pi[R_{t+1} + \gamma\, q_\pi(S_{t+1}, A_{t+1}) \mid S_t = s, A_t = a]$$

The two one-step "look-aheads" (backup diagrams):

$$v_\pi(s) = \sum_{a \in \mathcal{A}} \pi(a \mid s)\, q_\pi(s, a)$$

$$q_\pi(s, a) = \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a\, v_\pi(s')$$

Substituting one into the other gives the two-step versions:

$$v_\pi(s) = \sum_{a \in \mathcal{A}} \pi(a \mid s) \left( \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a\, v_\pi(s') \right)$$

$$q_\pi(s, a) = \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a \sum_{a' \in \mathcal{A}} \pi(a' \mid s')\, q_\pi(s', a')$$

**Matrix form** (via the induced MRP): $v_\pi = \mathcal{R}^\pi + \gamma \mathcal{P}^\pi v_\pi \;\Rightarrow\; v_\pi = (I - \gamma \mathcal{P}^\pi)^{-1} \mathcal{R}^\pi$

## 4. Optimality

**Optimal value functions:** the best achievable over all policies:

$$v_*(s) = \max_\pi v_\pi(s) \qquad q_*(s, a) = \max_\pi q_\pi(s, a)$$

An MDP is "solved" when you know $q_*$.

**Partial order on policies:** $\pi \ge \pi'$ if $v_\pi(s) \ge v_{\pi'}(s)$ for all $s$.

**Theorem.** For any MDP:

- there exists an optimal policy $\pi_*$ that is better than or equal to all others;
- all optimal policies achieve $v_*$ and $q_*$.

**Finding an optimal policy:** act greedily with respect to $q_*$:

$$\pi_*(a \mid s) = \begin{cases} 1 & \text{if } a = \arg\max_{a \in \mathcal{A}} q_*(s, a) \\ 0 & \text{otherwise} \end{cases}$$

There is always a deterministic optimal policy. Knowing $q_*$ immediately gives it, with no model needed. This is why model-free control (L5) learns $Q$.

### Bellman optimality equations

Replace the average over actions with a max:

$$v_*(s) = \max_a q_*(s, a)$$

$$q_*(s, a) = \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a\, v_*(s')$$

Combined:

$$v_*(s) = \max_a \left( \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a\, v_*(s') \right)$$

$$q_*(s, a) = \mathcal{R}_s^a + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a \max_{a'} q_*(s', a')$$

**Solving it:**

- The max makes it **non-linear**: no closed-form solution in general.
- Iterative methods: value iteration and policy iteration (L3, need a model); Q-learning and SARSA (L5, model-free).

## 5. Extensions (end of lecture, skim)

- Infinite and continuous MDPs: countably infinite states/actions, continuous states/actions (e.g. linear-quadratic regulator), continuous time (Hamilton–Jacobi–Bellman).
- **POMDPs:** hidden states; the agent keeps a **belief state** $b(h) = (\mathbb{P}[S_t = s^1 \mid H_t = h], \dots)$, a distribution over states given the history. A POMDP can be reduced to an (infinite) history tree or belief-state tree.
- Ergodic MDPs and average reward: for continuing tasks without discounting, optimize average reward per step instead.

## Quick self-check

1. What is the difference between $G_t$ and $v(s)$? *($G_t$ is a random sample of total reward; $v(s)$ is its expectation.)*
2. Why can an MRP's Bellman equation be solved directly but the optimality equation can't? *(The MRP one is linear; the max makes optimality non-linear.)*
3. Why does knowing $q_*$ (rather than $v_*$) give a policy without a model? *(Take $\arg\max_a q_*(s,a)$; with $v_*$ you'd need $\mathcal{P}$ and $\mathcal{R}$ to look one step ahead.)*
4. What does $\gamma$ control? *(How much future rewards count; effective horizon $\approx 1/(1-\gamma)$.)*

## Interview link

- **Formulating a process as an MDP:** state = measured process variables (Markov if it includes rates or recent history), action = control inputs, reward = tracking error, yield and cost terms, $\gamma$ close to 1 for slow processes with delayed effects.
- **If sensors miss key variables** (e.g. unmeasured concentration), it's a POMDP. Fix with stacked observations, an RNN, or an estimator: the "belief state" idea.
- $q_\pi$ vs $v_\pi$: $q$ lets you choose actions without a model, which is why Q-learning, DQN and SAC's critics learn $Q$.

# Silver Lecture 4 — Model-Free Prediction

*Companion notes, in slide order*

**Big picture:** estimate $v_\pi$ for an **unknown** MDP from experience alone. Two families: **Monte-Carlo (MC)** learns from complete episodes; **Temporal-Difference (TD)** learns from each step by bootstrapping. TD($\lambda$) blends them.

## 1. Monte-Carlo learning

- Learns directly from **complete episodes** of experience; no model.
- Idea: value = **mean return**.
- Only works for **episodic** MDPs (episodes must terminate).

**Goal:** learn $v_\pi$ from episodes $S_1, A_1, R_2, \dots, S_k \sim \pi$, using the actual return

$$G_t = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{T-1} R_T$$

and the empirical mean in place of the expectation $v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$.

**First-visit MC:** for each episode, at the *first* time $t$ state $s$ is visited: $N(s) \leftarrow N(s) + 1$, $S(s) \leftarrow S(s) + G_t$, $V(s) = S(s)/N(s)$. By the law of large numbers, $V(s) \to v_\pi(s)$.

**Every-visit MC:** same, but at *every* visit to $s$ in an episode.

*Example: Blackjack.* State = (current sum, dealer's showing card, usable ace). Policy: stick on 20 or 21, else twist. MC estimates the value surface after 10,000 and 500,000 episodes.

### Incremental mean

$$\mu_k = \mu_{k-1} + \frac{1}{k}\left(x_k - \mu_{k-1}\right)$$

**Incremental MC update:** after each episode, for each $S_t$ with return $G_t$:

$$N(S_t) \leftarrow N(S_t) + 1 \qquad V(S_t) \leftarrow V(S_t) + \frac{1}{N(S_t)}\left(G_t - V(S_t)\right)$$

For non-stationary problems, use a constant step size (forget old episodes):

$$V(S_t) \leftarrow V(S_t) + \alpha\left(G_t - V(S_t)\right)$$

General pattern for the whole course: **new estimate ← old estimate + step size × (target − old estimate).**

## 2. Temporal-Difference learning

- Learns from **incomplete episodes** by **bootstrapping**: updates a guess towards a guess.
- Model-free.

**TD(0):** update towards the estimated return $R_{t+1} + \gamma V(S_{t+1})$:

$$V(S_t) \leftarrow V(S_t) + \alpha\left( R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \right)$$

- **TD target:** $R_{t+1} + \gamma V(S_{t+1})$
- **TD error:** $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$

*Example: driving home.* MC only updates predictions once you arrive; TD updates each prediction as soon as you reach the next landmark.

### MC vs TD

| | MC | TD |
| --- | --- | --- |
| When it learns | After the episode ends | Online, every step |
| Incomplete sequences / continuing tasks | No | Yes |
| Target | $G_t$, unbiased | $R_{t+1} + \gamma V(S_{t+1})$, biased |
| Variance | High (depends on many random actions, transitions, rewards) | Low (one random action, transition, reward) |
| Sensitivity to initial value | Not sensitive | More sensitive |
| Uses Markov property | No; works in non-Markov environments too | Yes; more efficient in Markov environments |
| Convergence (tabular) | To $v_\pi$; good with function approximation | To $v_\pi$ (TD(0)); not always with function approximation |

Note: the *true* TD target $R_{t+1} + \gamma v_\pi(S_{t+1})$ is unbiased; it's the use of the estimate $V$ that introduces bias.

*Random walk example:* TD usually converges faster than MC for a range of $\alpha$.

### Batch MC and TD (AB example)

Replay a fixed set of $K$ episodes repeatedly:

- **MC** converges to the solution with **minimum mean-squared error** to the observed returns.
- **TD(0)** converges to the solution of the **maximum-likelihood Markov model** that best fits the data, i.e. it implicitly builds a model $\hat{\mathcal{P}}, \hat{\mathcal{R}}$ and solves it.

*AB example:* 8 episodes (A,0,B,0; then B,1 ×6; B,0 ×1). MC gives $V(A)=0$; TD gives $V(A)=0.75$.

### Unified view (backup diagrams)

| Method | Bootstraps? | Samples? |
| --- | --- | --- |
| Dynamic programming | Yes | No (full-width, needs model) |
| Monte-Carlo | No | Yes (full depth) |
| TD | Yes | Yes (one step) |
| Exhaustive search | No | No |

## 3. TD($\lambda$)

**n-step return:** look $n$ steps ahead, then bootstrap:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

$n=1$ is TD(0); $n=\infty$ is MC. Update: $V(S_t) \leftarrow V(S_t) + \alpha\left(G_t^{(n)} - V(S_t)\right)$.

**$\lambda$-return:** a weighted average of all $n$-step returns:

$$G_t^\lambda = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

**Forward-view TD($\lambda$):** $V(S_t) \leftarrow V(S_t) + \alpha\left(G_t^\lambda - V(S_t)\right)$. Like MC, it needs complete episodes.

**Backward-view TD($\lambda$)** with **eligibility traces** (credit both recently and frequently visited states):

$$E_0(s) = 0 \qquad E_t(s) = \gamma \lambda E_{t-1}(s) + \mathbf{1}(S_t = s)$$

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \qquad V(s) \leftarrow V(s) + \alpha\, \delta_t\, E_t(s) \quad \text{for all } s$$

- Online, updates every step, works on incomplete sequences.
- $\lambda = 0$: only the current state is updated → TD(0).
- $\lambda = 1$: credit deferred to the end → equivalent to every-visit MC (offline).
- Forward and backward views give the same total update (offline); "true online TD($\lambda$)" makes them exactly equivalent online.

## Quick self-check

1. What does "bootstrapping" mean? *(Updating an estimate using another estimate, e.g. $V(S_{t+1})$.)*
2. Why is TD lower variance than MC? *(Its target depends on one step of randomness, not a whole episode.)*
3. When would you prefer MC? *(Non-Markov states, or when you want unbiased targets, e.g. with function approximation.)*
4. What does $\lambda$ trade off? *(Bias vs variance, i.e. TD at $\lambda=0$ vs MC at $\lambda=1$.)*

## Interview link

- **Bias–variance trade-off of TD vs MC** is a classic question. Tie it to $\lambda$ and to PPO's **GAE**, which uses the same $\lambda$ idea for advantage estimates (typical $\lambda = 0.95$).
- **TD works on continuing tasks:** process control is usually continuing or long-horizon, so TD-based methods (actor-critic, SAC) fit better than pure MC.
- **The TD error $\delta_t$** reappears as the critic's learning signal in actor-critic (L7) and in PPO/SAC.

# Silver Lecture 5 — Model-Free Control

*Companion notes, in slide order*

**Big picture:** *optimise* the value function of an unknown MDP. Use generalised policy iteration with action values $Q$ and exploration. Main algorithms: **MC control**, **SARSA** (on-policy TD) and **Q-learning** (off-policy TD).

## 1. Introduction

Model-free control is used when the MDP is unknown but experience can be sampled, or when the MDP is known but too big to use except by sampling.

- **On-policy learning:** "learn on the job". Learn about policy $\pi$ from experience sampled from $\pi$.
- **Off-policy learning:** "look over someone's shoulder". Learn about target policy $\pi$ from experience sampled from a different behaviour policy $\mu$.

## 2. On-policy Monte-Carlo control

**Generalised policy iteration (recap):** evaluate the policy, then improve it greedily; repeat.

**Problem 1: greedy improvement over $V(s)$ needs a model:**

$$\pi'(s) = \arg\max_{a} \left( \mathcal{R}_s^a + \mathcal{P}_{ss'}^a V(s') \right)$$

**Fix:** use $Q(s,a)$ instead, which is model-free:

$$\pi'(s) = \arg\max_{a} Q(s, a)$$

**Problem 2: acting purely greedily stops exploring.** Fix with **$\epsilon$-greedy** exploration. With $m$ actions, every action has non-zero probability:

$$\pi(a \mid s) = \begin{cases} \epsilon/m + 1 - \epsilon & \text{if } a = \arg\max_{a'} Q(s, a') \\ \epsilon/m & \text{otherwise} \end{cases}$$

**Theorem:** for any $\epsilon$-greedy $\pi$, the $\epsilon$-greedy policy $\pi'$ with respect to $q_\pi$ is an improvement: $v_{\pi'}(s) \ge v_\pi(s)$.

**MC control:** every episode, evaluate $Q \approx q_\pi$ with MC, then improve with $\epsilon$-greedy.

**GLIE** (Greedy in the Limit with Infinite Exploration):

- every state–action pair is explored infinitely often: $\lim_{k\to\infty} N_k(s,a) = \infty$;
- the policy converges to greedy: $\lim_{k\to\infty} \pi_k(a \mid s) = \mathbf{1}\left(a = \arg\max_{a'} Q_k(s,a')\right)$.

Example: $\epsilon_k = 1/k$.

**GLIE MC control:** for each $(S_t, A_t)$ in episode $k$:

$$N(S_t, A_t) \leftarrow N(S_t, A_t) + 1 \qquad Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \frac{1}{N(S_t, A_t)}\left(G_t - Q(S_t, A_t)\right)$$

then $\epsilon \leftarrow 1/k$, $\pi \leftarrow \epsilon\text{-greedy}(Q)$. Converges to $q_*(s,a)$.

*Example:* Blackjack, optimal policy found with MC control.

## 3. On-policy TD learning: SARSA

Replace MC with TD in the control loop: lower variance, online, works with incomplete sequences.

**SARSA** (uses $S, A, R, S', A'$):

$$Q(S, A) \leftarrow Q(S, A) + \alpha\left( R + \gamma\, Q(S', A') - Q(S, A) \right)$$

where $A'$ is the action actually chosen by the current $\epsilon$-greedy policy. Every time step: evaluate with SARSA, improve with $\epsilon$-greedy.

**Algorithm:**

1. Initialise $Q(s,a)$ arbitrarily, $Q(\text{terminal}, \cdot) = 0$.
2. For each episode: initialise $S$; choose $A$ from $S$ using $\epsilon$-greedy($Q$).
3. Repeat each step: take $A$, observe $R, S'$; choose $A'$ from $S'$ using $\epsilon$-greedy($Q$); update $Q(S,A)$; $S \leftarrow S'$, $A \leftarrow A'$.

**Convergence:** SARSA converges to $q_*$ given a GLIE policy sequence and Robbins–Monro step sizes:

$$\sum_{t=1}^{\infty} \alpha_t = \infty \qquad \sum_{t=1}^{\infty} \alpha_t^2 < \infty$$

*Example: windy gridworld.*

**n-step SARSA:** target $q_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n})$.

**SARSA($\lambda$):** forward view uses the $q^\lambda$ return; backward view uses eligibility traces per state–action pair:

$$E_t(s,a) = \gamma \lambda E_{t-1}(s,a) + \mathbf{1}(S_t = s, A_t = a)$$

$$\delta_t = R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \qquad Q(s,a) \leftarrow Q(s,a) + \alpha\, \delta_t\, E_t(s,a)$$

Traces propagate a reward back along the whole recent path in one episode, rather than one step per episode.

## 4. Off-policy learning

Evaluate target policy $\pi(a \mid s)$ while following behaviour policy $\mu(a \mid s)$. Why:

- learn from observing humans or other agents;
- reuse experience from old policies;
- learn the **optimal** policy while following an **exploratory** policy;
- learn about multiple policies while following one.

**Importance sampling:**

$$\mathbb{E}_{X \sim P}[f(X)] = \mathbb{E}_{X \sim Q}\left[ \frac{P(X)}{Q(X)} f(X) \right]$$

- **Off-policy MC:** multiply the return by the ratio over the whole episode, $\frac{\pi(A_t|S_t)}{\mu(A_t|S_t)} \cdots \frac{\pi(A_T|S_T)}{\mu(A_T|S_T)}$. Extremely high variance; not practical.
- **Off-policy TD:** weight only one step: $V(S_t) \leftarrow V(S_t) + \alpha\left( \frac{\pi(A_t|S_t)}{\mu(A_t|S_t)} (R_{t+1} + \gamma V(S_{t+1})) - V(S_t) \right)$. Much lower variance.

### Q-learning

No importance sampling needed. Act with behaviour $A_t \sim \mu$, but bootstrap with the alternative successor action $A' \sim \pi$.

With target $\pi$ **greedy** and behaviour $\mu$ **$\epsilon$-greedy**, the target simplifies to a max:

$$Q(S, A) \leftarrow Q(S, A) + \alpha\left( R + \gamma \max_{a'} Q(S', a') - Q(S, A) \right)$$

**Theorem:** Q-learning control converges to $q_*(s,a)$ (tabular, all pairs visited, suitable step sizes).

*Example: cliff walking.* With $\epsilon$-greedy exploration:

- **Q-learning** learns the optimal path right along the cliff edge, but during training it sometimes falls off because exploration still happens there. Its *online* reward is worse.
- **SARSA** learns a safer path away from the edge, because its target accounts for the exploratory actions it will actually take.

## 5. Summary: DP vs TD

| Full backup (DP) | Sample backup (TD) |
| --- | --- |
| Iterative policy evaluation: $V(s) \leftarrow \mathbb{E}[R + \gamma V(S') \mid s]$ | TD learning: $V(S) \xleftarrow{\alpha} R + \gamma V(S')$ |
| Q-policy iteration: $Q(s,a) \leftarrow \mathbb{E}[R + \gamma Q(S',A') \mid s,a]$ | SARSA: $Q(S,A) \xleftarrow{\alpha} R + \gamma Q(S',A')$ |
| Q-value iteration: $Q(s,a) \leftarrow \mathbb{E}[R + \gamma \max_{a'} Q(S',a') \mid s,a]$ | Q-learning: $Q(S,A) \xleftarrow{\alpha} R + \gamma \max_{a'} Q(S',a')$ |

$x \xleftarrow{\alpha} y$ means $x \leftarrow x + \alpha(y - x)$.

## Quick self-check

1. Why learn $Q$ instead of $V$ for control? *(Greedy improvement over $Q$ needs no model.)*
2. SARSA vs Q-learning in one line? *(SARSA bootstraps from the action it will actually take (on-policy); Q-learning bootstraps from the best action (off-policy).)*
3. Why is Q-learning off-policy? *(Its target uses $\max_{a'}$, the greedy policy, regardless of the $\epsilon$-greedy behaviour.)*
4. What does the cliff example show? *(On-policy methods learn policies that are safe under their own exploration.)*

## Interview link

- **On- vs off-policy** is very likely to come up: PPO is on-policy (stable, sample-hungry); SAC and DQN are off-policy (replay buffer, sample-efficient). With an expensive simulator, off-policy is attractive.
- **The cliff example is a safety story:** in a real plant, exploration near constraint limits is dangerous. That's an argument for training in a digital twin, and for on-policy or constrained methods when exploring the real system.
- **Be able to write the Q-learning update and $\epsilon$-greedy from memory** for the Colab exercise (Drill D in your guide).

# Silver Lecture 7 — Policy Gradient Methods

*Companion notes, in slide order*

**Big picture:** instead of learning a value function and acting greedily, **parameterise the policy directly** and climb the gradient of expected return. Covers finite differences, the **score function**, the **policy gradient theorem**, **REINFORCE**, and **actor-critic** with baselines and advantages. This is the foundation for PPO and SAC.

## 1. Introduction

Previously: approximate $V$ or $Q$, derive the policy (e.g. $\epsilon$-greedy). Now approximate the policy itself:

$$\pi_\theta(s, a) = \mathbb{P}[a \mid s, \theta]$$

| | Value-based | Policy-based | Actor-critic |
| --- | --- | --- | --- |
| Learns | value function | policy | both |
| Policy | implicit ($\epsilon$-greedy) | explicit | explicit |

**Advantages of policy-based RL:**

- Better convergence properties (smooth changes to the policy).
- Effective in **high-dimensional or continuous action spaces**: no $\max_a$ over actions needed.
- Can learn **stochastic** policies (e.g. rock-paper-scissors; aliased gridworld where two states look identical).

**Disadvantages:**

- Typically converges to a **local** rather than global optimum.
- Evaluating a policy is often inefficient and **high variance**.

### Policy objective functions

Measure the quality of $\pi_\theta$:

- Episodic, start value: $J_1(\theta) = V^{\pi_\theta}(s_1) = \mathbb{E}_{\pi_\theta}[v_1]$
- Continuing, average value: $J_{avV}(\theta) = \sum_s d^{\pi_\theta}(s) V^{\pi_\theta}(s)$
- Continuing, average reward per step: $J_{avR}(\theta) = \sum_s d^{\pi_\theta}(s) \sum_a \pi_\theta(s,a) \mathcal{R}_s^a$

where $d^{\pi_\theta}(s)$ is the stationary distribution of the Markov chain under $\pi_\theta$.

**Optimisation:** find $\theta$ maximising $J(\theta)$. Gradient **ascent**:

$$\Delta \theta = \alpha \nabla_\theta J(\theta)$$

## 2. Finite-difference policy gradient

For each dimension $k$, perturb $\theta$ by $\epsilon$ in that dimension ($u_k$ = unit vector) and estimate:

$$\frac{\partial J(\theta)}{\partial \theta_k} \approx \frac{J(\theta + \epsilon u_k) - J(\theta)}{\epsilon}$$

Needs $n$ evaluations for $n$ parameters. Simple, noisy, inefficient, but works even for non-differentiable policies. *Example:* learning an AIBO robot's walking gait.

## 3. Monte-Carlo policy gradient

### Score function (likelihood-ratio trick)

Assume $\pi_\theta$ is differentiable when non-zero. Then:

$$\nabla_\theta \pi_\theta(s,a) = \pi_\theta(s,a) \frac{\nabla_\theta \pi_\theta(s,a)}{\pi_\theta(s,a)} = \pi_\theta(s,a)\, \nabla_\theta \log \pi_\theta(s,a)$$

The **score function** is $\nabla_\theta \log \pi_\theta(s,a)$.

**Softmax policy** (discrete actions), with features $\phi(s,a)$:

$$\pi_\theta(s,a) \propto e^{\phi(s,a)^\top \theta} \qquad \nabla_\theta \log \pi_\theta(s,a) = \phi(s,a) - \mathbb{E}_{\pi_\theta}[\phi(s, \cdot)]$$

**Gaussian policy** (continuous actions), mean $\mu(s) = \phi(s)^\top \theta$, fixed or learned variance $\sigma^2$:

$$a \sim \mathcal{N}(\mu(s), \sigma^2) \qquad \nabla_\theta \log \pi_\theta(s,a) = \frac{(a - \mu(s))\, \phi(s)}{\sigma^2}$$

*(This is how PPO and SAC handle continuous actions: the network outputs the mean and standard deviation of a Gaussian.)*

### One-step MDPs

Start in $s \sim d(s)$, take one action, get reward $r = \mathcal{R}_{s,a}$, end. Then:

$$J(\theta) = \mathbb{E}_{\pi_\theta}[r] \qquad \nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, r \right]$$

### Policy gradient theorem

Generalise to multi-step MDPs by replacing the immediate reward with the long-term value $Q^\pi(s,a)$. For any differentiable policy and any of the objectives $J_1$, $J_{avR}$, $\frac{1}{1-\gamma} J_{avV}$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, Q^{\pi_\theta}(s,a) \right]$$

**Intuition:** increase the log-probability of actions in proportion to how good they turned out to be.

### REINFORCE (Monte-Carlo policy gradient)

Use the sampled return $v_t$ (i.e. $G_t$) as an unbiased sample of $Q^{\pi_\theta}(s_t, a_t)$:

$$\Delta \theta_t = \alpha\, \nabla_\theta \log \pi_\theta(s_t, a_t)\, v_t$$

**Algorithm:** initialise $\theta$; for each episode $\{s_1, a_1, r_2, \dots, s_{T-1}, a_{T-1}, r_T\} \sim \pi_\theta$: for $t = 1$ to $T-1$, $\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(s_t, a_t) v_t$.

*Example: puck world.* Learning is smooth but slow; high variance from MC returns.

## 4. Actor-critic policy gradient

MC returns are high variance. Use a **critic** to estimate the action-value function:

$$Q_w(s,a) \approx Q^{\pi_\theta}(s,a)$$

- **Critic:** updates action-value parameters $w$ (policy evaluation, e.g. TD(0)).
- **Actor:** updates policy parameters $\theta$ in the direction the critic suggests.

Approximate policy gradient:

$$\nabla_\theta J(\theta) \approx \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, Q_w(s,a) \right] \qquad \Delta \theta = \alpha\, \nabla_\theta \log \pi_\theta(s,a)\, Q_w(s,a)$$

**QAC (action-value actor-critic)** with linear critic $Q_w(s,a) = \phi(s,a)^\top w$, every step:

1. Sample $r = \mathcal{R}_s^a$, $s' \sim \mathcal{P}_s^a$, $a' \sim \pi_\theta(s', \cdot)$.
2. $\delta = r + \gamma Q_w(s', a') - Q_w(s, a)$
3. Actor: $\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(s,a)\, Q_w(s,a)$
4. Critic: $w \leftarrow w + \beta\, \delta\, \phi(s,a)$
5. $a \leftarrow a'$, $s \leftarrow s'$

### Bias and compatible function approximation

An approximate critic can bias the gradient. The gradient is **exact** if the critic is *compatible*:

1. $\nabla_w Q_w(s,a) = \nabla_\theta \log \pi_\theta(s,a)$
2. $w$ minimises $\varepsilon = \mathbb{E}_{\pi_\theta}\left[ (Q^{\pi_\theta}(s,a) - Q_w(s,a))^2 \right]$

Then $\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}[\nabla_\theta \log \pi_\theta(s,a)\, Q_w(s,a)]$ exactly.

### Baselines and the advantage function

Subtract a **baseline** $B(s)$ to reduce variance **without changing the expectation**, because

$$\mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, B(s) \right] = \sum_s d^{\pi_\theta}(s)\, B(s)\, \nabla_\theta \sum_a \pi_\theta(s,a) = 0$$

A good baseline is the state value $B(s) = V^{\pi_\theta}(s)$, giving the **advantage function**:

$$A^{\pi_\theta}(s,a) = Q^{\pi_\theta}(s,a) - V^{\pi_\theta}(s)$$

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, A^{\pi_\theta}(s,a) \right]$$

"How much better is this action than average from this state?"

### TD error as an advantage estimate

The TD error with the true value function is an unbiased estimate of the advantage:

$$\delta^{\pi_\theta} = r + \gamma V^{\pi_\theta}(s') - V^{\pi_\theta}(s) \qquad \mathbb{E}_{\pi_\theta}[\delta^{\pi_\theta} \mid s, a] = A^{\pi_\theta}(s,a)$$

So use the TD error, which only needs a **state-value** critic $V_v(s)$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, \delta^{\pi_\theta} \right] \qquad \delta_v = r + \gamma V_v(s') - V_v(s)$$

### Critics and actors at different time-scales

The critic can use MC, TD(0), forward or backward TD($\lambda$). The actor can likewise use:

- MC: $\Delta \theta = \alpha (v_t - V_v(s_t)) \nabla_\theta \log \pi_\theta(s_t, a_t)$
- TD(0): $\Delta \theta = \alpha (r + \gamma V_v(s_{t+1}) - V_v(s_t)) \nabla_\theta \log \pi_\theta(s_t, a_t)$
- TD($\lambda$) with eligibility traces: $e_{t+1} = \lambda e_t + \nabla_\theta \log \pi_\theta(s,a)$, $\Delta \theta = \alpha\, \delta\, e_t$

### Natural policy gradient (skim)

The plain gradient depends on how the policy is parameterised. The **natural** gradient corrects for this using the Fisher information matrix $G_\theta$:

$$\nabla_\theta^{nat} J(\theta) = G_\theta^{-1} \nabla_\theta J(\theta) \qquad G_\theta = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(s,a)\, \nabla_\theta \log \pi_\theta(s,a)^\top \right]$$

With a compatible critic, the natural gradient is simply $w$, so the actor update is $\theta \leftarrow \theta + \alpha w$. *(TRPO is a practical descendant of this idea; PPO approximates TRPO.)*

## 5. Summary of policy gradient forms

All have the form $\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}[\nabla_\theta \log \pi_\theta(s,a) \times (\text{something})]$:

| Algorithm | "Something" |
| --- | --- |
| REINFORCE | $v_t$ (return) |
| Q actor-critic | $Q^w(s,a)$ |
| Advantage actor-critic | $A^w(s,a)$ |
| TD actor-critic | $\delta$ |
| TD($\lambda$) actor-critic | $\delta\, e$ |
| Natural actor-critic | $G_\theta^{-1} \nabla_\theta J(\theta) = w$ |

Each uses SGD on the actor; the critic uses policy evaluation (MC or TD) to estimate $Q^\pi$, $A^\pi$ or $V^\pi$.

## Quick self-check

1. Why use policy gradients instead of Q-learning for continuous actions? *(No $\max_a$ over a continuous space; the policy outputs actions directly.)*
2. What is the score function? *($\nabla_\theta \log \pi_\theta(s,a)$; the direction that makes action $a$ more likely.)*
3. Why does a baseline reduce variance without bias? *(Its expected contribution to the gradient is zero.)*
4. What does the critic do in actor-critic? *(Estimates values to give the actor a lower-variance learning signal, e.g. the TD error.)*

## Interview link

- **PPO** = advantage actor-critic + a clipped objective that limits how far the policy moves per update; advantages come from **GAE** (TD($\lambda$) for advantages).
- **SAC** = off-policy actor-critic with a Gaussian policy and an entropy bonus for exploration.
- **"Value-based vs policy-gradient?"** Value-based learns $Q$ and acts greedily (discrete actions); policy gradient optimises $\pi$ directly (continuous actions, stochastic policies); actor-critic combines both. Process control has continuous actions, so actor-critic methods fit.

# PPO and SAC — Spinning Up / Abbeel L4–L5 notes

*Fills the gap after Silver: the two algorithms most likely to come up*

**Big picture:** both are actor-critic methods (Silver L7) scaled up with neural networks. **PPO** is on-policy and limits how far each update moves the policy. **SAC** is off-policy, uses a replay buffer, and adds an entropy bonus for exploration. Both handle continuous actions.

## 1. PPO (Proximal Policy Optimization, 2017)

**Problem it solves:** vanilla policy gradient is sensitive to step size. One large update can collapse the policy, and because it's on-policy, a bad policy then collects bad data. TRPO fixed this with a KL-divergence constraint, but it's complex (second-order optimisation). PPO gets similar stability with a simple first-order objective.

### Probability ratio

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

$r_t > 1$: the new policy makes $a_t$ more likely than the old one did.

### Clipped surrogate objective

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[ \min\left( r_t(\theta)\, \hat{A}_t,\ \text{clip}\left(r_t(\theta), 1-\epsilon, 1+\epsilon\right) \hat{A}_t \right) \right]$$

- If $\hat{A}_t > 0$ (good action): increase its probability, but gain nothing once $r_t > 1 + \epsilon$.
- If $\hat{A}_t < 0$ (bad action): decrease its probability, but gain nothing once $r_t < 1 - \epsilon$.
- The $\min$ makes it a pessimistic bound: the policy can't profit from moving too far.
- Typical $\epsilon = 0.2$ (SB3 default `clip_range=0.2`).

### Advantage estimates: GAE

Generalised Advantage Estimation = TD($\lambda$) applied to advantages (Silver L4 + L7):

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) \qquad \hat{A}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l\, \delta_{t+l}$$

$\lambda = 0$: one-step TD advantage (low variance, more bias). $\lambda = 1$: MC advantage (high variance, unbiased). Typical $\lambda = 0.95$.

### Full loss (shared network version)

$$L(\theta) = \mathbb{E}_t\left[ L_t^{\text{CLIP}}(\theta) - c_1 \left( V_\theta(s_t) - V_t^{\text{target}} \right)^2 + c_2\, \mathcal{H}[\pi_\theta](s_t) \right]$$

Maximise: clipped policy objective − value-function error + entropy bonus (encourages exploration).

### Algorithm

1. Run the current policy for $T$ steps (in $N$ parallel environments); store $(s, a, r, \log \pi_{\text{old}})$.
2. Compute advantages $\hat{A}_t$ with GAE and value targets.
3. For $K$ epochs: split the batch into minibatches and do gradient ascent on $L$.
4. Discard the data (on-policy) and repeat.

### Key SB3 hyperparameters

| Parameter | Meaning | Default |
| --- | --- | --- |
| `n_steps` | Steps collected per env before each update | 2048 |
| `batch_size` | Minibatch size | 64 |
| `n_epochs` | Passes over each batch | 10 |
| `learning_rate` | Adam step size | 3e-4 |
| `gamma` | Discount | 0.99 |
| `gae_lambda` | GAE $\lambda$ | 0.95 |
| `clip_range` | $\epsilon$ | 0.2 |
| `ent_coef` | Entropy bonus $c_2$ | 0.0 |

### Continuous actions

The policy network outputs the mean of a Gaussian (and a learned log standard deviation); actions are sampled from it during training and the mean is used for deterministic evaluation. This is the Gaussian policy from Silver L7.

### Pros / cons

- **+** Stable, simple, robust default; works for discrete and continuous actions; easy to parallelise.
- **−** On-policy, so it throws data away after each update: **sample-inefficient**. Fine with a fast simulator; costly with an expensive one.

## 2. SAC (Soft Actor-Critic, 2018)

**Problem it solves:** on-policy methods waste samples; earlier off-policy continuous-control methods (DDPG) were brittle and sensitive to hyperparameters. SAC is off-policy and stable, and explores well via entropy.

### Maximum-entropy objective

Maximise reward **and** the policy's entropy:

$$J(\pi) = \sum_t \mathbb{E}_{(s_t, a_t) \sim \rho_\pi}\left[ r(s_t, a_t) + \alpha\, \mathcal{H}\left( \pi(\cdot \mid s_t) \right) \right] \qquad \mathcal{H}(\pi(\cdot \mid s)) = \mathbb{E}_{a \sim \pi}\left[ -\log \pi(a \mid s) \right]$$

- $\alpha$ (temperature) trades off reward vs randomness.
- The policy stays as random as possible while still doing well: built-in exploration, robustness, and it can keep several good options.
- Modern SAC **auto-tunes $\alpha$** to hit a target entropy (SB3 `ent_coef="auto"`).

### Components

- **Two Q-networks** $Q_{\phi_1}, Q_{\phi_2}$ (clipped double-Q): use the minimum to reduce overestimation.
- **Target Q-networks** $Q_{\phi_{\text{targ},1}}, Q_{\phi_{\text{targ},2}}$, updated by Polyak averaging $\phi_{\text{targ}} \leftarrow \tau \phi + (1-\tau)\phi_{\text{targ}}$ (DQN's target-network idea, softened).
- **Replay buffer** $\mathcal{D}$ (DQN idea; off-policy).
- **Stochastic policy** $\pi_\theta$: a Gaussian squashed through $\tanh$ so actions stay in bounds.

### Q-function target

$$y(r, s', d) = r + \gamma (1 - d)\left( \min_{j=1,2} Q_{\phi_{\text{targ},j}}(s', \tilde{a}') - \alpha \log \pi_\theta(\tilde{a}' \mid s') \right), \quad \tilde{a}' \sim \pi_\theta(\cdot \mid s')$$

Each Q-network minimises $\mathbb{E}_{\mathcal{D}}\left[ (Q_{\phi_i}(s,a) - y(r,s',d))^2 \right]$. $d = 1$ if $s'$ is terminal.

### Policy update

Maximise the soft Q-value of the policy's own actions:

$$\max_\theta\ \mathbb{E}_{s \sim \mathcal{D},\ \xi \sim \mathcal{N}}\left[ \min_{j=1,2} Q_{\phi_j}(s, \tilde{a}_\theta(s, \xi)) - \alpha \log \pi_\theta(\tilde{a}_\theta(s, \xi) \mid s) \right]$$

**Reparameterisation trick:** $\tilde{a}_\theta(s, \xi) = \tanh\left( \mu_\theta(s) + \sigma_\theta(s) \odot \xi \right)$, $\xi \sim \mathcal{N}(0, I)$. Lets gradients flow through the sampled action (lower variance than the score-function estimator).

### Algorithm

1. Observe $s$, sample $a \sim \pi_\theta(\cdot \mid s)$, step the environment, store $(s, a, r, s', d)$ in $\mathcal{D}$.
2. After each step (or every few): sample a minibatch from $\mathcal{D}$; update both Q-networks, the policy, $\alpha$ (if auto), and the target networks.
3. At test time, act with the mean action (deterministic).

### Related: DDPG and TD3 (Abbeel L5)

- **DDPG:** deterministic policy $\mu_\theta(s)$, Q-learning critic, replay buffer, target networks, exploration by adding noise to actions. Brittle; Q-values overestimate.
- **TD3:** fixes DDPG with (1) twin critics and the min target, (2) delayed policy updates, (3) target policy smoothing (noise on the target action). SAC uses the twin-critic idea too.

### Pros / cons

- **+** Sample-efficient (reuses data), stable, strong continuous-control performance, good exploration via entropy.
- **−** More components to tune and slower per step; continuous actions only (in standard form).

## 3. PPO vs SAC at a glance

| | PPO | SAC |
| --- | --- | --- |
| On/off-policy | On-policy | Off-policy (replay buffer) |
| Policy | Stochastic (Gaussian or categorical) | Stochastic, squashed Gaussian |
| Actions | Discrete or continuous | Continuous |
| Critic | $V(s)$ | Twin $Q(s,a)$ + targets |
| Exploration | Policy noise + optional entropy bonus | Entropy maximisation (auto-tuned $\alpha$) |
| Stability trick | Clipped ratio | Twin Q min, soft target updates |
| Sample efficiency | Lower | Higher |
| Best when | Simulator is cheap; want a robust default | Samples are expensive; continuous control |

## Quick self-check

1. What does PPO's clip prevent? *(Updates that move the policy too far from the one that collected the data.)*
2. Why can PPO not reuse old data, but SAC can? *(PPO's objective assumes data from the current/old policy; SAC learns Q off-policy from a replay buffer.)*
3. What does the entropy term in SAC do? *(Rewards randomness → exploration and robustness.)*
4. Why two Q-networks? *(Taking the min reduces Q-value overestimation.)*

## Interview link

- **"Which algorithm for a process-control twin?"** If the twin is fast: PPO as a stable baseline. If it's slow or expensive: SAC for sample efficiency. Continuous setpoints fit both; benchmark against PID.
- **Stability talking points:** normalise observations and rewards (`VecNormalize`), tune learning rate and `n_steps`/batch size, multiple seeds, watch episode return, value loss, entropy and (PPO) clip fraction / approx KL.
- **Action bounds:** SAC's $\tanh$ squashing and PPO's clipping to the action space map directly to actuator limits (valve 0–100%, heater power limits).

Sources: [Spinning Up — PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html), [Spinning Up — SAC](https://spinningup.openai.com/en/latest/algorithms/sac.html), [Stable-Baselines3 PPO docs](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html).